[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/cbm-colab-2026/blob/main/notebooks/2-3_%EA%B3%A0%EC%9E%A5%EC%A7%84%EB%8B%A8_%EB%B6%84%EB%A5%98%EB%AA%A8%EB%8D%B8.ipynb)

# 2-3 · 고장진단 — 어느 부품이 얼마나 나빠졌나

**⏱ 100분** · 모듈 2: 센서 → 특징 → 이상탐지 → **② 고장진단** → 판정 기준

## 🎯 이번 시간의 질문

> 과거 기록에 **정답(라벨)**이 있다면, 센서 특징으로 펌프 누설을 판정할 수 있을까요?

2-2의 결론: 정상 데이터만으로는 펌프 누설의 20%밖에 못 잡았고, 기준을 낮춰 86%를 잡으면 정상 사이클 3개 중 1개에도 경보가 울렸습니다. 게다가 경보는 **무엇이** 고장인지 알려 주지 않습니다.
하지만 과거 기록에 "이 사이클은 **누설 약함**이었다"는 정답이 있다면, 정답을 보면서 **센서 특징 → 상태**의 관계를 배울 수 있습니다. 이것이 **지도학습 분류**이고, CBM에서는 **고장진단**이라고 부릅니다.

| | 이상탐지 (2-2) | 고장진단 (2-3) |
|---|---|---|
| 학습 데이터 | 정상 데이터만 | 정상 + 여러 고장, **정답 포함** |
| 답하는 질문 | 평소와 다른가? | **어떤** 상태이고 **얼마나** 심한가? |
| 코드 | `model.fit(X)` | `model.fit(X, y)` |

**오늘의 대상:** 펌프 누설 상태 — 0 없음 · 1 약함 · 2 심함 (2-1 구분력 1.1, 2-2 이상탐지 20%)

**이번 시간에 익힐 패턴:** P5 모델 세 줄(`fit` → `predict` → 평가) 반복 · 혼동행렬과 상태별 탐지율 · 특징 중요도 막대그래프

In [ ]:
# ▶ 준비 셀 — 그대로 실행하세요. (실습 데이터와 한글 글꼴을 내려받습니다)
import os, urllib.request, warnings
REPO = "https://raw.githubusercontent.com/ralbu85/cbm-colab-2026/main/"
for path in ["data/hydraulic_labels.csv",
             "data/hydraulic_features.csv",
             "data/hydraulic_raw_4sensors.npz",
             "fonts/NanumGothic.ttf"]:
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(REPO + path, path)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_manager.fontManager.addfont("fonts/NanumGothic.ttf")
plt.rcParams.update({"font.family": "NanumGothic", "axes.unicode_minus": False,
                     "figure.figsize": (8, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore")
print("준비 완료!")

In [ ]:
# ▶ 그대로 실행하세요
labels = pd.read_csv('data/hydraulic_labels.csv')
features = pd.read_csv('data/hydraulic_features.csv')
data = labels.merge(features, on='cycle_id')
train = data[data['split'] == 'train']
test = data[data['split'] == 'test']

# 입력: 14개 센서의 60초 평균 특징 (2-2와 같음)
mean_cols = ['PS1_mean', 'PS2_mean', 'PS3_mean', 'PS4_mean', 'PS5_mean', 'PS6_mean', 'EPS1_mean',
             'FS1_mean', 'FS2_mean', 'TS1_mean', 'TS2_mean', 'TS3_mean', 'TS4_mean', 'VS1_mean']
print('학습:', len(train), '사이클 / 평가:', len(test), '사이클 / 입력 특징:', len(mean_cols), '개')

⚠️ **학습용과 평가용** — 2-1의 실험 일정표에서 본 것처럼 같은 상태 조합을 연달아 기록해서 이웃한 사이클은 거의 쌍둥이입니다. 그래서 **같은 상태 조합은 한쪽에만** 들어가도록 나눠 두었습니다(`split` 열). 평가용 336사이클은 모델이 **처음 보는 상태 조합**이므로, 평가 정확도는 "새 상황에서의 실력"입니다.

## 1. 입력(X)과 정답(y)

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
X_train = train[mean_cols]
y_train = train['pump_leakage']
X_test = test[mean_cols]
y_test = test['pump_leakage']
print(X_train.shape)
y_test.value_counts()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 학습 1,869개 × 특징 14개. 평가용은 누설 없음 90 · 약함 123 · 심함 123개입니다.

## 2. 결정나무 — 사람이 읽을 수 있는 규칙

결정나무는 "**유량이 6.5보다 작은가?**" 같은 **예/아니오 질문을 이어서** 답을 찾습니다. 어떤 센서로, 어떤 기준값에서 나눌지는 학습 데이터에서 자동으로 고릅니다. `max_depth=2`는 질문을 최대 두 번만 하라는 뜻입니다.

**P5 · 모델 세 줄** — 만들기 → 학습(`fit`) → 예측(`predict`), 그리고 정확도로 평가.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score

tree2 = DecisionTreeClassifier(max_depth=2, random_state=0)
tree2.fit(X_train, y_train)
pred2 = tree2.predict(X_test)
print('정확도:', round(accuracy_score(y_test, pred2) * 100, 1), '%')
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — 정확도 82.1%.

학습한 나무를 봅니다. 각 칸의 첫 줄이 질문, `samples`는 그 칸에 온 학습 사이클 수, `value`는 [없음, 약함, 심함] 개수, `class`는 그 칸의 판정입니다.

In [ ]:
# ▶ 그대로 실행하세요
plt.figure(figsize=(12, 5))
plot_tree(tree2, feature_names=mean_cols, class_names=['없음', '약함', '심함'],
          filled=True, impurity=False, fontsize=10)
plt.show()

💬 **결과 해설** — 나무가 스스로 고른 규칙입니다.

```
유량 FS1_mean > 6.54            → 누설 없음
6.44 < 유량 FS1_mean ≤ 6.54    → 약한 누설
유량 FS1_mean ≤ 6.44            → 심한 누설
```

"누설이 심할수록 유량이 줄어든다"는 물리 관계를 **기준값까지 정해서** 찾아냈습니다. 규칙을 사람이 읽고 검증할 수 있는 것이 결정나무의 큰 장점입니다.

## 3. 무엇을 틀렸나 — 혼동행렬과 상태별 탐지율

정확도 하나로는 **어떤 실수**를 했는지 모릅니다. 실제 × 예측 교차표(혼동행렬)를 봅니다. 대각선이 맞힌 것입니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
pd.crosstab(y_test, pred2, rownames=['실제'], colnames=['예측'])
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


각 행을 그 행의 합으로 나누면(`normalize='index'`) **실제 상태마다 몇 %를 제대로 찾았는지**가 나옵니다. 1-2·2-2의 **탐지율**을 상태별로 본 것입니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
(pd.crosstab(y_test, pred2, rownames=['실제'], colnames=['예측'], normalize='index') * 100).round(1)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — 대각선: 없음 88.9 · 약함 59.3 · **심함 100.0**(%).

💬 **결과 해설** — 심한 누설은 123개 **모두** 찾았지만, 약한 누설 123개 중 50개(41%)를 심한 누설로 봤습니다.
운영에서는 실수의 **종류**가 중요합니다. "약함 → 심함" 오판은 정비를 조금 서두르게 할 뿐이지만, "심함 → 약함·없음" 오판은 펌프 파손으로 이어질 수 있습니다.

## 4. 더 깊은 나무, 그리고 숲(Random Forest)

질문을 네 번까지 허용합니다. P5 세 줄에서 **숫자 하나만** 바꿉니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
tree4 = DecisionTreeClassifier(max_depth=4, random_state=0)
tree4.fit(X_train, y_train)
pred4 = tree4.predict(X_test)
print('정확도:', round(accuracy_score(y_test, pred4) * 100, 1), '%')
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 82.1% → **93.5%**. 질문이 늘면서 약함과 심함을 더 잘 가릅니다. 대신 규칙이 길어져 사람이 읽기는 어려워집니다.

**랜덤 포레스트**는 결정나무 **100그루**를 만들어 **투표**로 결정합니다. 각 나무는 학습 데이터와 특징의 **일부만 무작위로** 보고 학습하므로 서로 조금씩 다른 시각을 갖고, 한 나무의 실수를 다른 나무들이 보완합니다. 코드는 모델 이름만 다른 **같은 세 줄**입니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(random_state=0)
forest.fit(X_train, y_train)
pred_rf = forest.predict(X_test)
print('정확도:', round(accuracy_score(y_test, pred_rf) * 100, 1), '%')
(pd.crosstab(y_test, pred_rf, rownames=['실제'], colnames=['예측'], normalize='index') * 100).round(1)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — 정확도 97.3%. 대각선: 없음 98.9 · 약함 100.0 · 심함 93.5(%).

💬 **결과 해설** — 세 모델을 나란히 놓으면 흥미로운 점이 보입니다.

| 모델 | 정확도 | 약한 누설 탐지율 | **심한 누설 탐지율** |
|---|---|---|---|
| 결정나무 (깊이 2) | 82.1% | 59.3% | **100.0%** |
| 결정나무 (깊이 4) | 93.5% | 91.9% | 91.1% |
| 랜덤 포레스트 | 97.3% | 100.0% | 93.5% (8개 놓침) |

정확도가 가장 높은 랜덤 포레스트가 **가장 위험한 고장(심한 누설)은 8개 놓쳤습니다.** 정확도는 모든 칸을 똑같이 취급하기 때문입니다. "정확도가 높은 모델 = 현장에 가장 좋은 모델"은 아닙니다. 이 문제는 2-4에서 **판정 기준을 비용으로 정해** 해결합니다.

2-2의 이상탐지(누설 탐지율 20%)와 비교하면, **정답으로 학습한 효과**는 매우 큽니다.

## 5. 모델은 어떤 센서를 봤나 — 특징 중요도

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
importance = pd.Series(forest.feature_importances_, index=mean_cols).sort_values()
plt.barh(importance.index, importance.values)
plt.xlabel('특징 중요도')
plt.title('펌프 누설 진단에 쓰인 특징')
plt.show()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 유량 `FS1_mean`(약 0.42)이 가장 중요하고, 모터 전력 `EPS1_mean`(0.14), 압력 `PS3_mean`·`PS1_mean`이 뒤를 잇습니다. 2-1에서 펌프의 구분력 1·2위였던 두 특징(EPS1, FS1)입니다.
하나하나는 구분력 1 정도로 약하지만, 모델이 **여러 약한 특징을 조합**해 97%를 만들었습니다. 물리적으로 말이 되는 센서를 쓰고 있다는 것은 **모델을 믿을 근거**가 됩니다(중요도가 원인·결과를 증명하지는 않습니다).

### ▶ 진단 함수

과제에서 부품과 모델을 바꿔 가며 반복하도록, 위의 세 줄을 함수 하나로 묶었습니다.
`diagnose(모델, 부품, 특징목록)` → 모델을 학습(`fit`)하고 **학습 정확도와 평가 정확도**를 함께 돌려줍니다. 학습 정확도는 "배운 문제를 다시 풀었을 때", 평가 정확도는 "처음 보는 문제"의 점수입니다.

In [ ]:
# ▶ 그대로 실행하세요
def diagnose(model, part, cols=mean_cols):
    """model을 part 정답으로 학습하고, 학습·평가 정확도(%)를 돌려준다. 학습된 model은 그대로 다시 쓸 수 있다."""
    model.fit(train[cols], train[part])
    return pd.Series({'학습 정확도(%)': round(accuracy_score(train[part], model.predict(train[cols])) * 100, 1),
                      '평가 정확도(%)': round(accuracy_score(test[part], model.predict(test[cols])) * 100, 1)})

diagnose(RandomForestClassifier(random_state=0), 'pump_leakage')

✅ 평가 정확도 97.3%로 4절과 같으면 됩니다. 학습 정확도 100%는 "배운 문제는 다 맞힌다"는 뜻입니다.

---
## 6. 실습 과제 (40분)

### 📝 과제 1 · 네 부품 × 세 모델 진단표
표를 채우세요. 칸에는 **평가 정확도 (학습 정확도)**를 적습니다. **냉각기 줄은 P5 세 줄을 직접 입력**해서 구하고, 나머지는 `diagnose()`를 씁니다. 실행 전에 2-1의 구분력 표를 보고 어느 부품이 어려울지 예상해 보세요.

| 부품 (2-1 구분력) | 결정나무 깊이 2 | 결정나무 깊이 4 | 랜덤 포레스트 |
|---|---|---|---|
| 냉각기 (13.2) | | | |
| 펌프 (1.1) | 82.1 (90.5) | 93.5 (96.4) | 97.3 (100.0) |
| 밸브 (1.2) | | | |
| 축압기 (1.2) | | | |

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트 1 (냉각기, 직접 입력): tree_c = DecisionTreeClassifier(max_depth=2, random_state=0)
# tree_c.fit(X_train, train['cooler'])  →  accuracy_score(test['cooler'], tree_c.predict(X_test))
# 힌트 2 (나머지): diagnose(DecisionTreeClassifier(max_depth=2, random_state=0), 'valve')


✏️ **나의 답:** 어느 부품이 가장 어려웠나요? 학습 정확도와 평가 정확도의 차이가 가장 큰 칸은 어디이고, 그것은 무엇을 뜻하나요?

### 📝 과제 2 · 좋은 특징 하나로 밸브 진단 살리기
2-1 과제 4의 새 특징 **"9~10초 압력 평균"**(구분력 8.9)을 입력에 추가하면 밸브 진단이 좋아질까요?

1. 원신호에서 특징 만들기 → `raw = np.load('data/hydraulic_raw_4sensors.npz')`, `data['PS1_peak9'] = raw['PS1'][:, 900:1000].mean(axis=1)`
2. 새 열이 들어간 `data`로 학습/평가를 **다시 나누기** → `train = data[data['split'] == 'train']`, `test = ...` (`diagnose()`는 이 `train`, `test`를 씁니다)
3. 특징 목록에 추가 → `cols_plus = mean_cols + ['PS1_peak9']`, `diagnose(모델, 'valve', cols_plus)`

| 밸브 진단 입력 | 결정나무 깊이 4 | 랜덤 포레스트 |
|---|---|---|
| 평균 특징 14개 | 43.2 (70.0) | 23.2 (100.0) |
| + PS1_peak9 | | |

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: 위 1~3단계를 차례로 입력합니다. 혼동행렬은 m = DecisionTreeClassifier(max_depth=4, random_state=0); diagnose(m, 'valve', cols_plus) 후
# pd.crosstab(test['valve'], m.predict(test[cols_plus]))


✏️ **나의 답:**

### 📝 과제 3 · 생각해 보기 — 현장에 어떤 모델을 넣을까?
펌프 누설 진단에 4절의 세 모델 중 하나를 골라야 합니다. 정확도, **심한 누설 탐지율**, 규칙을 사람이 읽을 수 있는지를 근거로 고르고 이유를 쓰세요. 랜덤 포레스트가 놓친 8개를 줄이는 방법도 떠올려 보세요.

✏️ **나의 답:**

---
## ✅ 정리

| 배운 것 | 핵심 |
|---|---|
| 지도학습 분류 | 정답(y)이 있는 과거 데이터로 `fit(X, y)` → 새 사이클의 상태를 `predict(X)` |
| 결정나무 | 예/아니오 질문의 연속. 규칙을 **읽을 수 있다** (유량 ≤ 6.44 → 심한 누설) |
| 랜덤 포레스트 | 나무 100그루의 투표. 약한 특징 여러 개를 조합 (펌프 97%, 축압기 88%) |
| 평가 | 정확도 + **상태별 탐지율**. 정확도가 높아도 가장 위험한 고장을 놓칠 수 있다 |
| 과적합 | 학습 100% · 평가 23% (밸브) → 처음 보는 데이터로 평가해야 한다 |
| 가장 큰 교훈 | 입력에 정보가 없으면 어떤 모델도 못 맞힌다. **좋은 특징 > 복잡한 모델** (밸브 43% → 98%) |

다음 질문: 랜덤 포레스트가 놓친 심한 누설 8개를, 판정 기준을 바꿔 줄일 수 있을까요? 기준은 얼마로 정해야 할까요? → **2-4 · 판정 기준을 비용으로 정하기**